# Vision Modality: Train / Evaluate on the Full Dataset

Runs on the real cached OpenFace tables (no mock data). Every labelled video with a cache file is used: it is split by video into train / val / test, the model is trained, and the final cell reports training, validation and test accuracy plus the per-chunk and per-video predictions.

## Imports and configuration

In [1]:
import sys, os, json, random
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '../..'))
if project_root not in sys.path:
    sys.path.append(project_root)
os.environ["LIBGL_ALWAYS_SOFTWARE"] = "1"

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from src.models.vision_encoder import VisionModel


# ---------------- config ----------------
DATA_DIR     = os.path.expanduser("../../data")                 # folder with the .mp4 files (labels from folder names)
CACHE_DIR    = os.path.join(DATA_DIR, "openface_cache")      # OpenFace .pkl cache from the EDA notebook
OUT_DIR      = "runs/vision"

PROJ_DIM     = 128
HIDDEN_DIM   = 128
NUM_LAYERS   = 2
DROPOUT      = 0.3

CHUNK_LEN    = 240               # frames per chunk (8 s @ 30 fps)
STRIDE       = 180               # frames between chunk starts (2 s overlap)
CONF_THRESH  = 0.75

EPOCHS       = 20
BATCH_SIZE   = 16
LR           = 1e-3
WEIGHT_DECAY = 1e-2
USE_AMP      = False             # mixed precision, GPU only
SEED         = 0
NUM_WORKERS  = 0
# ----------------------------------------

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

def seed_all(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_all(SEED)

Device: cuda


## Dataset code
Discovery, video-level splitting, OpenFace cache loading, normalisation and sliding-window chunking.

In [2]:
import hashlib
import os
import glob

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset

AU_R = [1, 2, 4, 5, 6, 7, 9, 10, 12, 14, 15, 17, 20, 23, 25, 26, 45]
AU_C = [1, 2, 4, 5, 6, 7, 9, 10, 12, 14, 15, 17, 20, 23, 25, 26, 28, 45]

DEFAULT_FEATURES = (
    [f"gaze_{e}_{a}" for e in (0, 1) for a in "xyz"]
    + ["gaze_angle_x", "gaze_angle_y"]
    + [f"pose_{c}" for c in ("Tx", "Ty", "Tz", "Rx", "Ry", "Rz")]
    + [f"AU{i:02d}_r" for i in AU_R]
    + [f"AU{i:02d}_c" for i in AU_C]
)  # 49 dims

LIE_TOKENS = {"deception", "deceptive", "lie", "lies", "false"}
TRUTH_TOKENS = {"truth", "truthful", "true"}


def feature_cache_path_for(video_path, cache_dir):
    """Identical to the notebook's cache naming."""
    file_hash = hashlib.sha256(os.path.abspath(video_path).encode("utf-8")).hexdigest()
    stem = os.path.splitext(os.path.basename(video_path))[0]
    return os.path.join(cache_dir, f"{stem}_{file_hash}.pkl")


def label_from_path(video_path):
    """1 = deceptive, 0 = truthful, from the folder names (then the filename)."""
    parts = [p.lower() for p in os.path.normpath(video_path).split(os.sep)[:-1]]
    parts.append(os.path.splitext(os.path.basename(video_path))[0].lower())
    for p in reversed(parts):
        tokens = set(p.replace("-", "_").split("_")) | {p}
        if tokens & LIE_TOKENS:
            return 1
        if tokens & TRUTH_TOKENS:
            return 0
    return None


def discover_videos(data_dir, cache_dir):
    """Return [(video_path, cache_path, label)] for videos that have a cache and a label."""
    paths = sorted(glob.glob(os.path.join(data_dir, "**", "*.mp4"), recursive=True))
    items, no_cache, no_label = [], 0, 0
    for p in paths:
        label = label_from_path(p)
        cache = feature_cache_path_for(p, cache_dir)
        if label is None:
            no_label += 1
        elif not os.path.exists(cache):
            no_cache += 1
        else:
            items.append((p, cache, label))
    if no_cache or no_label:
        print(f"[discover] skipped {no_cache} videos without cache, {no_label} without label")
    return items


def split_videos(items, val_frac=0.15, test_frac=0.15, seed=0):
    """Stratified video-level split."""
    rng = np.random.RandomState(seed)
    train, val, test = [], [], []
    for lab in sorted({it[2] for it in items}):
        grp = [it for it in items if it[2] == lab]
        rng.shuffle(grp)
        n_val = max(1, int(round(len(grp) * val_frac)))
        n_test = max(1, int(round(len(grp) * test_frac)))
        val += grp[:n_val]
        test += grp[n_val:n_val + n_test]
        train += grp[n_val + n_test:]
    return train, val, test


def load_feature_table(cache_path, feature_cols, conf_thresh=0.75):
    """-> feats (N, D) float32 with NaN -> 0, valid (N,) bool."""
    df = pd.read_pickle(cache_path)
    df.columns = df.columns.str.strip()
    missing = [c for c in feature_cols if c not in df.columns]
    if missing:
        raise KeyError(f"{os.path.basename(cache_path)} lacks columns: {missing[:5]}...")
    feats = np.nan_to_num(df[feature_cols].to_numpy(dtype=np.float32))
    conf = pd.to_numeric(df["confidence"], errors="coerce").fillna(0.0).to_numpy()
    valid = conf >= conf_thresh
    if "success" in df.columns:
        valid &= df["success"].to_numpy() == 1
    return feats, valid


def compute_stats(items, feature_cols, conf_thresh=0.75):
    """Per-feature mean/std over valid frames of the given (train) videos."""
    s, ss, n = 0.0, 0.0, 0
    for _, cache, _ in items:
        x, v = load_feature_table(cache, feature_cols, conf_thresh)
        x = x[v].astype(np.float64)
        s += x.sum(0)
        ss += (x ** 2).sum(0)
        n += len(x)
    mean = s / max(n, 1)
    std = np.sqrt(np.maximum(ss / max(n, 1) - mean ** 2, 1e-12))
    std[std < 1e-6] = 1.0  # constant columns
    return mean.astype(np.float32), std.astype(np.float32)


def chunk_starts(n, chunk_len, stride, min_len):
    if n <= chunk_len:
        return [0]
    starts = list(range(0, n - chunk_len + 1, stride))
    tail = starts[-1] + stride
    if tail < n and n - tail >= min_len:
        starts.append(tail)  # partial tail chunk, padded in __getitem__
    return starts


class VisionChunkDataset(Dataset):
    """Items: dict(x=(T, D) float, length=int, label=int, video_idx=int)."""

    def __init__(self, items, mean, std, feature_cols=DEFAULT_FEATURES, chunk_len=240,
                 stride=180, min_len=30, conf_thresh=0.75):
        self.items = items
        self.mean, self.std = mean, std
        self.chunk_len = chunk_len
        self.tables = []   # per video: (normalised feats with invalid frames zeroed)
        self.index = []    # (video_idx, start)
        for vi, (_, cache, _) in enumerate(items):
            x, valid = load_feature_table(cache, feature_cols, conf_thresh)
            if len(x) == 0:
                self.tables.append(None)
                continue
            x = (x - mean) / std
            x[~valid] = 0.0
            self.tables.append(x)
            for s in chunk_starts(len(x), chunk_len, stride, min_len):
                self.index.append((vi, s))
        self.feature_dim = len(feature_cols)

    def __len__(self):
        return len(self.index)

    def __getitem__(self, i):
        vi, s = self.index[i]
        x = self.tables[vi][s:s + self.chunk_len]
        length = len(x)
        out = np.zeros((self.chunk_len, self.feature_dim), dtype=np.float32)
        out[:length] = x
        return {
            "x": torch.from_numpy(out),
            "length": torch.tensor(length, dtype=torch.long),
            "label": torch.tensor(self.items[vi][2], dtype=torch.long),
            "video_idx": torch.tensor(vi, dtype=torch.long),
        }

## Data
Video-level stratified split, so overlapping chunks never leak across splits. Normalisation statistics come from the training videos only.

In [3]:
n_mp4 = len(glob.glob(os.path.join(DATA_DIR, "**", "*.mp4"), recursive=True))
items = discover_videos(DATA_DIR, CACHE_DIR)
assert items, "No labelled videos with cached OpenFace tables found. Check DATA_DIR / CACHE_DIR."
print(f"{len(items)} / {n_mp4} videos used")

train_v, val_v, test_v = split_videos(items, seed=SEED)
mean, std = compute_stats(train_v, DEFAULT_FEATURES, CONF_THRESH)
ds_kw = dict(mean=mean, std=std, feature_cols=DEFAULT_FEATURES, chunk_len=CHUNK_LEN,
             stride=STRIDE, conf_thresh=CONF_THRESH)
train_ds, val_ds, test_ds = (VisionChunkDataset(v, **ds_kw) for v in (train_v, val_v, test_v))

summary = pd.DataFrame({
    "videos": [len(train_v), len(val_v), len(test_v)],
    "deceptive": [sum(i[2] == 1 for i in v) for v in (train_v, val_v, test_v)],
    "truthful":  [sum(i[2] == 0 for i in v) for v in (train_v, val_v, test_v)],
    "chunks": [len(train_ds), len(val_ds), len(test_ds)],
}, index=["train", "val", "test"])
display(summary)
print(f"T={CHUNK_LEN} D={train_ds.feature_dim}")

dl_kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=DEVICE == "cuda")
train_dl      = DataLoader(train_ds, shuffle=True, drop_last=len(train_ds) > BATCH_SIZE, **dl_kw)
train_eval_dl = DataLoader(train_ds, shuffle=False, **dl_kw)   # same data, no shuffling, for metrics/predictions
val_dl        = DataLoader(val_ds, **dl_kw)
test_dl       = DataLoader(test_ds, **dl_kw)

[discover] skipped 751 videos without cache, 0 without label
828 / 1579 videos used


,videos,deceptive,truthful,chunks
train,580,580,0,719
val,124,124,0,161
test,124,124,0,162


T=240 D=49


## Model

In [5]:
model = VisionModel(
    train_ds.feature_dim, num_classes=2, proj_dim=PROJ_DIM,
    hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS, dropout=DROPOUT,
).to(DEVICE)
print(f"params: {sum(p.numel() for p in model.parameters()):,}")

opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
use_amp = USE_AMP and DEVICE == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
crit = nn.CrossEntropyLoss()

b = next(iter(train_dl))
with torch.no_grad():
    print("batch x:", tuple(b["x"].shape), "-> logits:", tuple(model(b["x"].to(DEVICE), b["length"].to(DEVICE)).shape))

params: 666,980
batch x: (16, 240, 49) -> logits: (16, 2)


## Evaluation helpers
`run_split` does one eval-mode pass over a split and returns chunk-level and video-level metrics plus the predictions. Video-level prediction = mean chunk probability per source video. Deceptive is the positive class.

In [6]:
def f1_binary(y, p):
    tp = int(((y == 1) & (p == 1)).sum())
    fp = int(((y == 0) & (p == 1)).sum())
    fn = int(((y == 1) & (p == 0)).sum())
    return 2 * tp / max(2 * tp + fp + fn, 1)

@torch.no_grad()
def run_split(model, loader, ds, name):
    model.eval()
    probs, ys, vids = [], [], []
    for b in loader:
        logits = model(b["x"].to(DEVICE), b["length"].to(DEVICE))
        probs.append(torch.softmax(logits.float(), -1)[:, 1].cpu())
        ys.append(b["label"]); vids.append(b["video_idx"])
    probs, ys, vids = torch.cat(probs).numpy(), torch.cat(ys).numpy(), torch.cat(vids).numpy()
    pred = (probs >= 0.5).astype(int)

    chunk_df = pd.DataFrame({
        "split": name,
        "video": [os.path.basename(ds.items[v][0]) for v in vids],
        "chunk_start": [ds.index[i][1] for i in range(len(ds))],
        "label": ys, "prob_deceptive": probs, "pred": pred,
    })
    video_df = (chunk_df.groupby("video", sort=False)
                .agg(split=("split", "first"), label=("label", "first"),
                     prob_deceptive=("prob_deceptive", "mean"), n_chunks=("pred", "size"))
                .reset_index())
    video_df["pred"] = (video_df["prob_deceptive"] >= 0.5).astype(int)

    m = {
        "chunk_acc": float((chunk_df.pred == chunk_df.label).mean()),
        "chunk_f1": f1_binary(chunk_df.label.to_numpy(), chunk_df.pred.to_numpy()),
        "video_acc": float((video_df.pred == video_df.label).mean()),
        "video_f1": f1_binary(video_df.label.to_numpy(), video_df.pred.to_numpy()),
        "n_chunks": len(chunk_df), "n_videos": len(video_df),
    }
    return m, chunk_df, video_df

## Training
Each epoch ends with an eval-mode pass over the full train set and the full validation set. The best epoch (by validation chunk F1) is kept.

In [8]:
os.makedirs(OUT_DIR, exist_ok=True)
CKPT = os.path.join(OUT_DIR, "best.pt")
keys = ["chunk_acc", "chunk_f1", "video_acc", "video_f1"]
history = {"loss": [], **{f"train_{k}": [] for k in keys}, **{f"val_{k}": [] for k in keys}}
best, best_ep = -1.0, 0

for ep in range(1, EPOCHS + 1):
    model.train()
    tot, n = 0.0, 0
    for b in tqdm(train_dl, desc=f"epoch {ep}/{EPOCHS}", leave=False):
        x, l, y = b["x"].to(DEVICE), b["length"].to(DEVICE), b["label"].to(DEVICE)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE, enabled=use_amp):
            loss = crit(model(x, l), y)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update()
        tot += loss.item() * len(y); n += len(y)
    sched.step()

    tr, _, _ = run_split(model, train_eval_dl, train_ds, "train")
    va, _, _ = run_split(model, val_dl, val_ds, "val")
    history["loss"].append(tot / n)
    for k in keys:
        history[f"train_{k}"].append(tr[k]); history[f"val_{k}"].append(va[k])
    print(f"ep {ep:02d} loss={tot / n:.4f} | chunk acc train={tr['chunk_acc']:.3f} val={va['chunk_acc']:.3f} "
          f"| video acc train={tr['video_acc']:.3f} val={va['video_acc']:.3f}")

    if va["chunk_f1"] > best:
        best, best_ep = va["chunk_f1"], ep
        torch.save({"model": model.state_dict(), "mean": mean, "std": std,
                    "features": DEFAULT_FEATURES,
                    "config": dict(proj_dim=PROJ_DIM, hidden_dim=HIDDEN_DIM,
                                   num_layers=NUM_LAYERS, dropout=DROPOUT,
                                   chunk_len=CHUNK_LEN, stride=STRIDE)}, CKPT)
print(f"best epoch: {best_ep} (val chunk F1 {best:.3f})")

epoch 1/20:   0%|          | 0/44 [00:00<?, ?it/s]

ep 01 loss=0.0000 | chunk acc train=1.000 val=1.000 | video acc train=1.000 val=1.000


epoch 2/20:   0%|          | 0/44 [00:00<?, ?it/s]

ep 02 loss=0.0000 | chunk acc train=1.000 val=1.000 | video acc train=1.000 val=1.000


epoch 3/20:   0%|          | 0/44 [00:00<?, ?it/s]

ep 03 loss=0.0000 | chunk acc train=1.000 val=1.000 | video acc train=1.000 val=1.000


... (28 outputs left out)

epoch 18/20:   0%|          | 0/44 [00:00<?, ?it/s]

ep 18 loss=0.0000 | chunk acc train=1.000 val=1.000 | video acc train=1.000 val=1.000


epoch 19/20:   0%|          | 0/44 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
axes[0].plot(history["loss"], marker="o"); axes[0].set_title("Train loss"); axes[0].set_xlabel("epoch")
axes[1].plot(history["train_chunk_acc"], marker="o", label="train"); axes[1].plot(history["val_chunk_acc"], marker="o", label="val")
axes[1].set_title("Chunk accuracy"); axes[1].set_xlabel("epoch"); axes[1].set_ylim(0, 1.05); axes[1].legend()
axes[2].plot(history["train_video_acc"], marker="o", label="train"); axes[2].plot(history["val_video_acc"], marker="o", label="val")
axes[2].set_title("Video accuracy"); axes[2].set_xlabel("epoch"); axes[2].set_ylim(0, 1.05); axes[2].legend()
plt.tight_layout(); plt.show()

## Final results and predictions (best checkpoint)
One eval-mode pass over the entire train, validation and test sets.

In [ ]:
model.load_state_dict(torch.load(CKPT, map_location=DEVICE, weights_only=False)["model"])

rows, chunk_dfs, video_dfs = {}, [], []
for name, loader, ds in (("train", train_eval_dl, train_ds), ("val", val_dl, val_ds), ("test", test_dl, test_ds)):
    m, cdf, vdf = run_split(model, loader, ds, name)
    rows[name] = m; chunk_dfs.append(cdf); video_dfs.append(vdf)

results = pd.DataFrame(rows).T
results[["n_chunks", "n_videos"]] = results[["n_chunks", "n_videos"]].astype(int)
display(results.round(4))

chunk_preds = pd.concat(chunk_dfs, ignore_index=True)
video_preds = pd.concat(video_dfs, ignore_index=True)
chunk_preds.to_csv(os.path.join(OUT_DIR, "chunk_predictions.csv"), index=False)
video_preds.to_csv(os.path.join(OUT_DIR, "video_predictions.csv"), index=False)
results.to_csv(os.path.join(OUT_DIR, "results.csv"))
print(f"saved predictions + results to {OUT_DIR}/")
video_preds.head(10)